# Лекција 17 - Креирање Локалних AI Агената са Foundry Local и Qwen

У овом бележнику правите **локалног инжењерског асистента** који ради у потпуности на вашем рачунару. Ниједна обрада у облаку се не користи у ни једном тренутку. Асистент може:

1. **Позивати алате** преко Qwen функцијског позива кроз Foundry Local.
2. **Наводити и читати фајлове** унутар пројектног директоријума са ограниченим приступом.
3. **Анализирати код** са једноставним метриком.
4. **Претраживати документацију** са локалним RAG (Chroma).
5. **Користити локални MCP сервер** (пребачено без грешке ако није конфигурисан).

Код агента изгледа скоро идентично као код лекција о облаку — једино што се крајња тачка клијента помера са облака на `localhost`.


## Подешавање

Пре покретања овог бележника:

1. **Инсталирајте Microsoft Foundry Local** (погледајте [документацију](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) за свој ОС).
2. **Преузмите и покрените Qwen модел:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Инсталирајте доле наведене Python пакете.

Све ради локално. Машина са ~8 ГБ РАМ-а је реалан минимум.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Повежите се са Foundry Local

`FoundryLocalManager` преузима модел ако је потребно, покреће локалну услугу и даје нам **ендпоинт који је компатибилан са OpenAI-јем**. Затим упућујемо стандардни OpenAI SDK на њега. API кључ је локални плејсхолдер — није укључен ниједан облачни акредитив.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Локални алати (операције са фајловима у пијеску)

Направимо мали пример пројекта на диску, а затим дефинишемо алате који су ограничени на коријен тог пројекта. Провјера пијеска је битна чак и локално: алат који чита произвољне путеве ради са дозволама вашег корисника и може приступити свему што и ви.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Локални RAG са Chromom

Уграђујемо мали скуп исечака из документације у локалну Chroma колекцију. Chroma ради у процесу и чува векторе на диску — нема сервера, нема облака. Алат `search_docs` преузима најрелевантније исечке за упит.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Петља за позивање алата

Сада региструјемо алате са моделом користећи OpenAI шему алата и покрећемо стандардну петљу за позивање алата — модел тражи алат, ми га извршавамо локално, враћамо резултат и понављамо док модел не произведе коначни одговор. Поуздано позивање функција у Qwen-у је оно што омогућава да ово функционише уређају.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Локални MCP (опционо)

MCP је транспорт, а не cloud услуга — MCP сервер може да ради као локални процес преко `stdio`. Ћелија испод приказује како бисте се повезали са локалним MCP сервером ако имате конфигурисаног (на пример сервер за фајл систем). Он лепо прескаче када `LOCAL_MCP_COMMAND` није подешен, па ће свезак још увек функционисати од почетка до краја без њега.

Безбедносна напомена: локални MCP сервер ради са дозволама вашег корисника. Ограничите га на директоријум пројекта и верификујте његове излазне податке пре него што их употребите.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Резиме

Направили сте инжењерског асистента који ради у потпуности на вашем рачунару:

- **Foundry Local** је служио **Qwen** модел иза OpenAI-компатибилног ендпоинта — тако да код агента одговара лекцијама из облака.
- **Sandboxed алати** дали су агенту приступ фајловима и анализу кода без напуштања директоријума пројекта.
- **Chroma** је пружила **локални RAG** преко документације.
- **Local MCP** показао је како поново користити MCP екосистем офлајн.

Није коришћена никаква облачна инференција у било ком тренутку.

### Изазов

Проширите локалног агента да:

1. **Ради са више MCP сервера** — повежите фајл системски сервер и гит сервер и омогућите агенту да бира између њих.
2. **Користи локалну меморију** — сачувајте кратку историју разговора на диск тако да асистент памти претходне кораке након поновног покретања бележнице.
3. **Подржи локалну мулти-агентну оркестрацију** — додајте другог локалног агента (нпр. рецензента) и омогућите двојици да сарађују на задатку.

У следећој лекцији ћете научити како да обезбедите распоређене AI агенте.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
